# PSA-MT — Notebook 1: NLLB-200 Transfer Learning

**Purpose:** train and evaluate the NLLB-200 distilled model separately from mT5.

This notebook is intentionally **not** a full project notebook. It handles:
1. shared data preparation (only if the shared split does not already exist),
2. NLLB zero-shot baseline,
3. NLLB few-shot fine-tuning,
4. checkpointing after **every epoch**,
5. per-epoch metrics/logs,
6. final BLEU / SacreBLEU / chrF++ / COMET evaluation,
7. saving the trained model for later inference/deployment.

**Important:** run this notebook first if `shared_artifacts/data_processed/` does not yet exist. The second notebook will reuse the exact same saved train/dev/test files.

## 1. Install dependencies

In [5]:
!pip -q install -U transformers datasets accelerate sentencepiece sacrebleu mlflow evaluate
!pip install -q ipywidgets

In [2]:
# ==========================
# Suppress warnings & logs
# Run this as the FIRST cell
# ==========================

import os
import warnings
import logging

# Silence Python warnings
warnings.filterwarnings("ignore")

# Silence Hugging Face warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Silence MLflow Git warnings
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# Reduce TensorFlow logs (harmless if TensorFlow isn't used)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Silence Hugging Face Hub progress/warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Silence Python logging
logging.getLogger().setLevel(logging.ERROR)

# Silence specific libraries
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

print("✅ Warnings and logs suppressed.")

✅ Warnings and logs suppressed.


## 1b. All imports (everything below this cell only *uses* these -- no more scattered imports)

In [3]:
from pathlib import Path
import os, json, time, random, math, shutil, subprocess, sys, inspect

import numpy as np
import pandas as pd
import torch
import sacrebleu
import mlflow

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback,
)

# NOTE: `comet` is deliberately NOT imported here -- it's installed and imported in its
# own isolated cell later, only once you actually reach the COMET section. Importing it
# here would fail before that install ever runs.

print("All core imports loaded.")

import warnings
warnings.simplefilter("ignore")

All core imports loaded.


In [8]:
# =========================================================================
# Rebuild everything COMET needs, WITHOUT retraining -- safe to paste as one block.
# =========================================================================
from pathlib import Path
import os, json, time, random, math, shutil, subprocess, sys, inspect
import numpy as np
import pandas as pd
import torch
import sacrebleu
import mlflow
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

IN_COLAB = os.path.exists("/var/colab/hostname")
INPUT_DIR = Path.cwd()
OUTPUT_ROOT = INPUT_DIR / "PSA-MT-Outputs"
SHARED_DIR = OUTPUT_ROOT / "shared_artifacts"
DATA_DIR = SHARED_DIR / "data_processed"
MODELS_DIR = OUTPUT_ROOT / "models"
BASE_MODELS_DIR = MODELS_DIR / "base_pretrained"
FT_MODELS_DIR = MODELS_DIR / "fine_tuned"
RESULTS_DIR = OUTPUT_ROOT / "results"
LOGS_DIR = OUTPUT_ROOT / "logs"
MLFLOW_DIR = OUTPUT_ROOT / "mlflow"

SEED = 42
random.seed(SEED)
MAX_LEN = 128
EPOCHS = 5
TRAIN_BATCH = 32
EVAL_BATCH = 32
GRAD_ACCUM = 2
FREEZE_ENCODER = True
NLLB_FREEZE_LAYERS = 6
USE_GRADIENT_CHECKPOINTING = True
CHECKPOINT_KEEP = 2
USE_AUGMENTATION = True
AUGMENT_DROP_PROB = 0.10
RESUME_CHECKPOINT = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": "kin_Latn"}
NLLB_DIRECTIONS = ["English_to_Ekegusii", "Kiswahili_to_Ekegusii"]
NLLB_OUTPUT = FT_MODELS_DIR / "nllb"
BASE_NLLB = BASE_MODELS_DIR / "nllb"

def word_dropout_noise(text, drop_prob=AUGMENT_DROP_PROB, rng=None):
    rng = rng or random
    words = str(text).split()
    kept = [w for w in words if rng.random() > drop_prob]
    return " ".join(kept) if kept else str(text)

def load_direction(slug, fewshot=None, augment=True):
    out = {}
    for sp in ["train", "dev", "test"]:
        df = pd.read_csv(DATA_DIR/f"{slug}.{sp}.csv").dropna(subset=["src_text", "tgt_text"])
        if sp == "train":
            if fewshot is not None:
                df = df.sample(min(fewshot, len(df)), random_state=SEED).reset_index(drop=True)
            else:
                df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
            if augment and USE_AUGMENTATION and "Ekegusii" in slug:
                extra = df.copy(); rng = random.Random(SEED)
                extra["src_text"] = extra["src_text"].apply(lambda x: word_dropout_noise(x, rng=rng))
                df = pd.concat([df, extra], ignore_index=True)
        out[sp] = Dataset.from_pandas(df, preserve_index=False)
    return DatasetDict(out)

def evaluate_model(model, tok, ds, src, tgt, batch=8):
    model.eval(); preds=[]; refs=[]; srcs=[]
    tok.src_lang = NLLB_CODE[src]
    for i in range(0, len(ds), batch):
        texts = list(ds["src_text"][i:i+batch])
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        forced = tok.convert_tokens_to_ids(NLLB_CODE[tgt])
        gen = model.generate(**enc, forced_bos_token_id=forced, max_length=MAX_LEN)
        preds.extend(tok.batch_decode(gen, skip_special_tokens=True))
        refs.extend(ds["tgt_text"][i:i+batch]); srcs.extend(texts)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    return preds, refs, srcs, {"bleu": round(bleu,2), "chrf": round(chrf,2)}

print("All variables/functions reconstructed. NLLB_DIRECTIONS:", NLLB_DIRECTIONS)
print("NLLB_OUTPUT:", NLLB_OUTPUT)
for d in NLLB_DIRECTIONS:
    exists = (NLLB_OUTPUT/d/"best"/"config.json").exists()
    print(f"  {d}: best/ checkpoint exists = {exists}")

DEVICE: cuda
All variables/functions reconstructed. NLLB_DIRECTIONS: ['English_to_Ekegusii', 'Kiswahili_to_Ekegusii']
NLLB_OUTPUT: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb
  English_to_Ekegusii: best/ checkpoint exists = True
  Kiswahili_to_Ekegusii: best/ checkpoint exists = True


## 2. Hardware check

In [5]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: training on CPU will be much slower. Use a hosted GPU runtime.")

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM GB: 79.25


## 3. Experiment tracking (MLflow)

Satisfies "set up experiment tracking (W&B or MLflow)." Every training run below logs its
hyperparameters and metrics here -- both the zero-shot baseline and the few-shot fine-tuned
runs, so you can compare them later without re-reading printed cell output.

In [6]:
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DIR}/mlflow.db")
mlflow.set_experiment("psa-mt-nllb")
print("MLflow tracking to:", f"sqlite:///{MLFLOW_DIR}/mlflow.db")
print("View later with: mlflow ui --backend-store-uri", f"sqlite:///{MLFLOW_DIR}/mlflow.db")

MLflow tracking to: sqlite:////home/jovyan/PSA_MT_Project/PSA-MT-Outputs/mlflow/mlflow.db
View later with: mlflow ui --backend-store-uri sqlite:////home/jovyan/PSA_MT_Project/PSA-MT-Outputs/mlflow/mlflow.db


## 4. Create/load the **single shared split**

The split is made once at the **underlying PSA-row level**, then reused by both NLLB and mT5. This prevents train/test leakage and makes model comparisons fair.

If the files already exist, this cell **does not regenerate them**.

In [7]:

DIRECTIONS = [
    ("English","Kiswahili"), ("Kiswahili","English"),
    ("English","Ekegusii"), ("Ekegusii","English"),
    ("Kiswahili","Ekegusii"), ("Ekegusii","Kiswahili")
]

# CHANGE ONLY THIS if your cleaned combined dataset is stored elsewhere.
RAW_DATA = INPUT_DIR / "kenyan_psa_multilingual_dataset.csv"  # <-- CONFIRM this matches your exact filename in NLP_Translation

def read_any(path):
    if path.suffix.lower() == ".csv": return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx",".xls"]: return pd.read_excel(path)
    if path.suffix.lower() == ".json": return pd.read_json(path)
    raise ValueError("Use CSV/XLSX/JSON")

def detect_col(df, names):
    lower={str(c).strip().lower():c for c in df.columns}
    for n in names:
        if n.lower() in lower: return lower[n.lower()]
    for c in df.columns:
        s=str(c).lower()
        if any(n.lower() in s for n in names): return c
    return None

def make_shared_split():
    if not RAW_DATA.exists():
        raise FileNotFoundError(f"Put the cleaned combined dataset here: {RAW_DATA}")
    df=read_any(RAW_DATA).reset_index(drop=True)
    en=detect_col(df,["English","English Text","English_PSA"])
    sw=detect_col(df,["Kiswahili","Swahili","Kiswahili Text"])
    ek=detect_col(df,["Ekegusii","Ekegusii Text","Gusii"])
    dom=detect_col(df,["Domain","PSA Domain","Category"])
    pid=detect_col(df,["PSA_ID","ID","id"])
    if not all([en,sw,ek]):
        raise ValueError(f"Could not detect all language columns. Found: {list(df.columns)}")

    rng=np.random.default_rng(SEED)
    idx=np.arange(len(df)); rng.shuffle(idx)
    n=len(df); ntr=int(.80*n); nd=int(.10*n)
    split=np.empty(n,dtype=object)
    split[idx[:ntr]]="train"; split[idx[ntr:ntr+nd]]="dev"; split[idx[ntr+nd:]]="test"
    df["_split"]=split

    colmap={"English":en,"Kiswahili":sw,"Ekegusii":ek}
    summary=[]
    for src,tgt in DIRECTIONS:
        sub=df.dropna(subset=[colmap[src],colmap[tgt]]).copy()
        keep=[colmap[src],colmap[tgt],"_split"] + ([dom] if dom else []) + ([pid] if pid else [])
        sub=sub[keep]
        ren={colmap[src]:"src_text",colmap[tgt]:"tgt_text"}
        if dom: ren[dom]="Domain"
        if pid: ren[pid]="PSA_ID"
        sub=sub.rename(columns=ren)
        slug=f"{src}_to_{tgt}"
        for sp in ["train","dev","test"]:
            part=sub[sub["_split"]==sp].drop(columns="_split").reset_index(drop=True)
            part.to_csv(DATA_DIR/f"{slug}.{sp}.csv",index=False,encoding="utf-8-sig")
            summary.append({"direction":slug,"split":sp,"n":len(part)})
    meta={"seed":SEED,"ratios":{"train":.8,"dev":.1,"test":.1},
          "source_file":str(RAW_DATA),"created_at":time.strftime("%Y-%m-%d %H:%M:%S")}
    (SHARED_DIR/"split_manifest.json").write_text(json.dumps(meta,indent=2))
    pd.DataFrame(summary).to_csv(RESULTS_DIR/"direction_split_summary.csv",index=False)
    print(pd.DataFrame(summary).pivot(index="direction",columns="split",values="n"))

required=DATA_DIR/"English_to_Kiswahili.train.csv"
if required.exists():
    print("Shared split already exists. Reusing it.")
else:
    make_shared_split()

Shared split already exists. Reusing it.


## 5. Shared split integrity check

In [8]:
manifest=json.loads((SHARED_DIR/"split_manifest.json").read_text())
print(json.dumps(manifest,indent=2))
for slug in ["English_to_Kiswahili","Kiswahili_to_English","English_to_Ekegusii","Ekegusii_to_English","Kiswahili_to_Ekegusii","Ekegusii_to_Kiswahili"]:
    for sp in ["train","dev","test"]:
        p=DATA_DIR/f"{slug}.{sp}.csv"
        assert p.exists(), p
print("All six directions have saved train/dev/test files.")

{
  "seed": 42,
  "ratios": {
    "train": 0.8,
    "dev": 0.1,
    "test": 0.1
  },
  "source_file": "/home/jovyan/PSA_MT_Project/kenyan_psa_multilingual_dataset.csv",
  "created_at": "2026-08-01 21:19:16"
}
All six directions have saved train/dev/test files.


## 6. NLLB model configuration

**Training:** 4 epochs.  
**Low-resource setting:** maximum 2,000 training examples per direction + frozen encoder.  
**Checkpoint policy:** a checkpoint is saved at the end of **every epoch** and retained on Drive.  
**Recovery policy:** rerun the notebook with `RESUME_CHECKPOINT` pointing to the latest checkpoint.

**Scope, by request: NLLB runs ONLY English→Ekegusii and Kiswahili→Ekegusii.** No
English<->Kiswahili in this notebook anymore -- every NLLB result in this project targets Ekegusii.

**Ekegusii via repurposed token:** NLLB-200 has no native Ekegusii language code. To still get
an NLLB result on these two directions, we repurpose an existing, unused NLLB token (`kin_Latn`,
Kinyarwanda -- a related East African Bantu language) as a stand-in tag, then fine-tune NLLB to
map that tag to Ekegusii text. **This is a workaround, not native support** -- keep that caveat
attached to every result from this notebook in your report. Zero-shot is expected to be
poor/meaningless (the model has never seen Ekegusii under any tag) -- the fine-tuned result is
the one that actually tests the technique.

In [9]:

NLLB_NAME="facebook/nllb-200-distilled-600M"
NLLB_CODE={
    "English":"eng_Latn",
    "Kiswahili":"swh_Latn",
    # Ekegusii has NO native NLLB-200 language code. We repurpose "kin_Latn"
    # (Kinyarwanda -- a geographically/typologically related East African Bantu
    # language) as a stand-in tag, then fine-tune NLLB to associate that tag with
    # Ekegusii text instead. This is a documented low-resource MT workaround, NOT
    # native Ekegusii support -- label every result from this direction accordingly
    # in the report (e.g. "NLLB fine-tuned via repurposed kin_Latn token").
    "Ekegusii":"kin_Latn",
}

# By request: NLLB runs ONLY the two Ekegusii directions -- no English<->Kiswahili.
# "English" and "Kiswahili" still need real NLLB_CODE entries above because they're the
# SOURCE side of these two directions; only Ekegusii itself is the repurposed token.
NLLB_DIRECTIONS = ["English_to_Ekegusii","Kiswahili_to_Ekegusii"]
print("NLLB will run:", NLLB_DIRECTIONS)
print("NOTE: both directions use a REPURPOSED token (kin_Latn) for Ekegusii, not native NLLB support.")

# Persistent Hugging Face cache on Drive
os.environ["HF_HOME"]=str(OUTPUT_ROOT/"hf_cache")
os.environ["TRANSFORMERS_CACHE"]=str(OUTPUT_ROOT/"hf_cache"/"transformers")
os.makedirs(os.environ["TRANSFORMERS_CACHE"],exist_ok=True)

BASE_NLLB=BASE_MODELS_DIR/"nllb-200-distilled-600M"
NLLB_OUTPUT=FT_MODELS_DIR/"nllb"

print("Base model cache:", BASE_NLLB)
print("Fine-tuned checkpoints:", NLLB_OUTPUT)

NLLB will run: ['English_to_Ekegusii', 'Kiswahili_to_Ekegusii']
NOTE: both directions use a REPURPOSED token (kin_Latn) for Ekegusii, not native NLLB support.
Base model cache: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/base_pretrained/nllb-200-distilled-600M
Fine-tuned checkpoints: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb


## 7. Download once and save the pretrained NLLB model for later reuse

This is **not retrained**. It is the original pretrained checkpoint. The fine-tuned model is saved separately.

For deployment later, you normally load the **best fine-tuned checkpoint**, not the base model.

In [10]:
def load_or_save_base_nllb():
    if (BASE_NLLB/"config.json").exists():
        tok=AutoTokenizer.from_pretrained(BASE_NLLB)
        model=AutoModelForSeq2SeqLM.from_pretrained(BASE_NLLB)
        print("Loaded pretrained NLLB from Drive.")
    else:
        tok=AutoTokenizer.from_pretrained(NLLB_NAME)
        model=AutoModelForSeq2SeqLM.from_pretrained(NLLB_NAME)
        BASE_NLLB.mkdir(parents=True,exist_ok=True)
        tok.save_pretrained(BASE_NLLB)
        model.save_pretrained(BASE_NLLB)
        print("Downloaded and saved pretrained NLLB to Drive.")
    return tok,model

base_tok,base_model=load_or_save_base_nllb()
del base_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

Loaded pretrained NLLB from Drive.


## 8. Data + low-resource augmentation

In [11]:
def word_dropout_noise(text, drop_prob=AUGMENT_DROP_PROB, rng=None):
    rng = rng or random
    words = str(text).split()
    kept = [w for w in words if rng.random() > drop_prob]
    return " ".join(kept) if kept else str(text)


def load_direction(slug, fewshot=None, augment=True):   # <-- FIXED: was fewshot=FEWSHOT_N
    out = {}
    for sp in ["train", "dev", "test"]:
        df = pd.read_csv(DATA_DIR/f"{slug}.{sp}.csv").dropna(subset=["src_text", "tgt_text"])
        if sp == "train":
            if fewshot is not None:                       # <-- FIXED: guard against None
                df = df.sample(min(fewshot, len(df)), random_state=SEED).reset_index(drop=True)
            else:
                df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle, keep all rows
            # Same low-resource augmentation technique as the mT5 notebook, applied here too
            # since the Ekegusii-via-proxy-token directions are genuinely low-resource for NLLB.
            if augment and USE_AUGMENTATION and "Ekegusii" in slug:
                extra = df.copy(); rng = random.Random(SEED)
                extra["src_text"] = extra["src_text"].apply(lambda x: word_dropout_noise(x, rng=rng))
                df = pd.concat([df, extra], ignore_index=True)
        out[sp] = Dataset.from_pandas(df, preserve_index=False)
    return DatasetDict(out)


def freeze_encoder(model, num_layers=None):
    """num_layers=None -> freeze the WHOLE encoder (original behavior).
    num_layers=N -> freeze only the bottom N encoder layers (partial-freezing technique)."""
    encoder = model.get_encoder()
    if num_layers is None:
        for p in encoder.parameters():
            p.requires_grad = False
        return
    layers = encoder.layers if hasattr(encoder, "layers") else encoder.block
    for i, layer in enumerate(layers):
        if i < num_layers:
            for p in layer.parameters():
                p.requires_grad = False


def preprocess_nllb(tokenizer, src, tgt):
    tokenizer.src_lang = NLLB_CODE[src]
    tokenizer.tgt_lang = NLLB_CODE[tgt]
    def fn(batch):
        return tokenizer(batch["src_text"], text_target=batch["tgt_text"], max_length=MAX_LEN, truncation=True)
    return fn

## 9. Per-epoch checkpoint + metric callback

The callback writes `epoch_metrics.csv` after each evaluation and keeps the Hugging Face `checkpoint-*` directory for each epoch. If the hosted notebook session disconnects, the latest checkpoint can be resumed.

In [12]:
class EpochArtifactCallback(TrainerCallback):
    def __init__(self, run_dir):
        self.run_dir=Path(run_dir); self.run_dir.mkdir(parents=True,exist_ok=True)
        self.rows=[]
    def on_evaluate(self,args,state,control,metrics=None,**kwargs):
        if metrics:
            row={"epoch":state.epoch,"step":state.global_step,**metrics}
            self.rows.append(row)
            pd.DataFrame(self.rows).to_csv(self.run_dir/"epoch_metrics.csv",index=False)
            (self.run_dir/f"epoch_{int(round(state.epoch)):02d}_metrics.json").write_text(json.dumps(row,indent=2,default=str))

def compute_metrics_builder(tok):
    def compute(eval_pred):
        preds,labels=eval_pred
        if isinstance(preds,tuple): preds=preds[0]
        labels=np.where(labels!=-100,labels,tok.pad_token_id)
        p=tok.batch_decode(preds,skip_special_tokens=True)
        r=tok.batch_decode(labels,skip_special_tokens=True)
        return {
            "bleu":sacrebleu.corpus_bleu(p,[r]).score,
            "chrf":sacrebleu.corpus_chrf(p,[r],word_order=2).score
        }
    return compute

# ---- transformers version compatibility -------------------------------------
# Newer transformers releases renamed Trainer's `tokenizer=` kwarg to
# `processing_class=`. This picks whichever one the installed version wants,
# so training doesn't crash if `pip install -U transformers` pulled a newer release.

def build_seq2seq_trainer(model, args, train_ds, eval_ds, collator, compute_metrics,
                          tokenizer, callbacks=None):
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=collator, compute_metrics=compute_metrics)
    if callbacks:
        kw["callbacks"] = callbacks
    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    kw["processing_class" if "processing_class" in params else "tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kw)

In [13]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


## 10. NLLB zero-shot baseline

In [14]:
def evaluate_model(model, tok, ds, src, tgt, batch=8):
    model.eval(); preds=[]; refs=[]; srcs=[]
    tok.src_lang = NLLB_CODE[src]
    for i in range(0, len(ds), batch):
        texts = list(ds["src_text"][i:i+batch])
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        forced = tok.convert_tokens_to_ids(NLLB_CODE[tgt])
        gen = model.generate(**enc, forced_bos_token_id=forced, max_length=MAX_LEN)
        preds.extend(tok.batch_decode(gen, skip_special_tokens=True))
        refs.extend(ds["tgt_text"][i:i+batch]); srcs.extend(texts)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    return preds, refs, srcs, {"bleu": round(bleu, 2), "chrf": round(chrf, 2)}


zero_results = []
for direction in NLLB_DIRECTIONS:
    src, tgt = direction.split("_to_")
    ds = load_direction(direction, fewshot=None, augment=False)   # explicit now, was implicit before
    tok = AutoTokenizer.from_pretrained(BASE_NLLB)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_NLLB).to(DEVICE)
    t0 = time.time()
    preds, refs, srcs, metrics = evaluate_model(model, tok, ds["test"], src, tgt, batch=EVAL_BATCH)  # was default 8
    metrics.update({"model": "NLLB", "setting": "zero-shot", "direction": direction, "runtime_min": (time.time()-t0)/60})
    zero_results.append(metrics)
    with mlflow.start_run(run_name=f"nllb-zeroshot-{direction}"):
        mlflow.log_params({"model": "NLLB", "setting": "zero-shot", "direction": direction})
        mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
    print(metrics)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

pd.DataFrame(zero_results).to_csv(RESULTS_DIR/"nllb_zero_shot.csv", index=False)

{'bleu': 1.9, 'chrf': 19.32, 'model': 'NLLB', 'setting': 'zero-shot', 'direction': 'English_to_Ekegusii', 'runtime_min': 1.5312549074490864}
{'bleu': 0.97, 'chrf': 18.58, 'model': 'NLLB', 'setting': 'zero-shot', 'direction': 'Kiswahili_to_Ekegusii', 'runtime_min': 1.5167837023735047}


## 11. NLLB few-shot fine-tuning — 4 epochs

Each direction is a separate run. The encoder is frozen to reduce overfitting and compute.

**Checkpoint folders:**
`models/fine_tuned/nllb/<direction>/checkpoint-...`

**Best model:**
`models/fine_tuned/nllb/<direction>/best/`

The notebook records:
- training start/end/runtime,
- parameters,
- epoch metrics,
- checkpoint paths,
- final test metrics.

In [15]:
from tqdm.auto import tqdm

class TqdmProgressCallback(TrainerCallback):
    """A visual progress bar, completely separate from EpochArtifactCallback's clean
    text logging -- this just tracks step-by-step progress visually, it doesn't print
    or save anything, so it can't duplicate or conflict with the other callback."""
    def on_train_begin(self, args, state, control, **kwargs):
        self.bar = tqdm(total=state.max_steps, desc="Training", unit="step")

    def on_step_end(self, args, state, control, **kwargs):
        self.bar.n = state.global_step
        self.bar.refresh()

    def on_train_end(self, args, state, control, **kwargs):
        self.bar.close()

In [16]:
def train_nllb(direction):
    src, tgt = direction.split("_to_")

    # ---- FULL dataset now (was 50%) -- domain-stratified with frac=1.0 for consistency ----
    dsd = load_direction(direction, fewshot=None, augment=False)
    train_df = dsd["train"].to_pandas()

    if "Domain" in train_df.columns:
        sampled = (train_df.groupby("Domain", group_keys=False)
                  .apply(lambda g: g.sample(frac=1.0, random_state=SEED)))
    else:
        sampled = train_df.sample(frac=1.0, random_state=SEED)

    if USE_AUGMENTATION and "Ekegusii" in direction:
        extra = sampled.copy()
        rng = random.Random(SEED)
        extra["src_text"] = extra["src_text"].apply(lambda x: word_dropout_noise(x, rng=rng))
        sampled = pd.concat([sampled, extra], ignore_index=True)

    dsd["train"] = Dataset.from_pandas(sampled.reset_index(drop=True), preserve_index=False)
    print(f"[{direction}] train examples: {len(dsd['train'])} (FULL dataset + augmentation)")

    tok = AutoTokenizer.from_pretrained(BASE_NLLB)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_NLLB).to(DEVICE)
    freeze_encoder(model, num_layers=NLLB_FREEZE_LAYERS)

    enc = dsd.map(preprocess_nllb(tok, src, tgt), batched=True,
                  remove_columns=dsd["train"].column_names)

    run_dir = NLLB_OUTPUT / direction
    run_dir.mkdir(parents=True, exist_ok=True)

    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        learning_rate=3e-5,
        per_device_train_batch_size=TRAIN_BATCH,   # was hardcoded 32 -- now from config
        per_device_eval_batch_size=EVAL_BATCH,     # was hardcoded 32 -- now from config
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,                   # was hardcoded 5 -- now from config
        warmup_ratio=0.1,
        weight_decay=0.0,
        optim="adamw_torch",
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        fp16=(DEVICE == "cuda"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=CHECKPOINT_KEEP,
        load_best_model_at_end=True,
        metric_for_best_model="chrf",
        greater_is_better=True,
        logging_steps=25,
        report_to=["mlflow"],
        gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        disable_tqdm=True,
    )

    cb = EpochArtifactCallback(run_dir)
    trainer = build_seq2seq_trainer(
        model=model, args=args, train_ds=enc["train"], eval_ds=enc["dev"],
        collator=DataCollatorForSeq2Seq(tok, model=model),
        compute_metrics=compute_metrics_builder(tok), tokenizer=tok, callbacks=[cb],
    )

    from transformers.trainer_callback import PrinterCallback
    trainer.remove_callback(PrinterCallback)
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except ImportError:
        pass

    print(f"\n=== Training {direction} -- logs also saved to {run_dir}/training_steps.csv ===\n")

    with mlflow.start_run(run_name=f"nllb-full-{EPOCHS}ep-{direction}"):
        mlflow.log_params({"model": "NLLB", "direction": direction, "epochs": EPOCHS,
                           "train_fraction": 1.0, "freeze_layers": NLLB_FREEZE_LAYERS,
                           "learning_rate": 3e-5, "train_batch": TRAIN_BATCH, "grad_accum": GRAD_ACCUM})
        t0 = time.time()
        trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
        runtime = (time.time() - t0) / 60

        best_dir = run_dir / "best"
        trainer.save_model(best_dir)
        tok.save_pretrained(best_dir)

        preds, refs, srcs, metrics = evaluate_model(trainer.model, tok, dsd["test"], src, tgt)
        final = {**metrics, "model": "NLLB", "setting": f"full-{EPOCHS}epochs", "direction": direction,
                "runtime_min": round(runtime, 2), "epochs": EPOCHS, "train_fraction": 1.0,
                "freeze_encoder": FREEZE_ENCODER}
        mlflow.log_metrics({k: v for k, v in final.items() if isinstance(v, (int, float))})

    pd.DataFrame([final]).to_csv(run_dir / "final_test_metrics.csv", index=False)
    print("\nFINAL:", final)
    print(f"Full step-by-step log saved at: {run_dir}/training_steps.csv")
    print(f"Full epoch-by-epoch log saved at: {run_dir}/epoch_metrics.csv")

    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final


nllb_final = []
for direction in NLLB_DIRECTIONS:
    nllb_final.append(train_nllb(direction))
pd.DataFrame(nllb_final).to_csv(RESULTS_DIR / "nllb_fewshot_final.csv", index=False)

[English_to_Ekegusii] train examples: 46832 (FULL dataset + augmentation)


Map:   0%|          | 0/46832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2927 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]


=== Training English_to_Ekegusii -- logs also saved to /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/English_to_Ekegusii/training_steps.csv ===


FINAL: {'bleu': 11.23, 'chrf': 35.91, 'model': 'NLLB', 'setting': 'full-5epochs', 'direction': 'English_to_Ekegusii', 'runtime_min': 40.84, 'epochs': 5, 'train_fraction': 1.0, 'freeze_encoder': True}
Full step-by-step log saved at: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/English_to_Ekegusii/training_steps.csv
Full epoch-by-epoch log saved at: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/English_to_Ekegusii/epoch_metrics.csv
[Kiswahili_to_Ekegusii] train examples: 46832 (FULL dataset + augmentation)


Map:   0%|          | 0/46832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2927 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]


=== Training Kiswahili_to_Ekegusii -- logs also saved to /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/Kiswahili_to_Ekegusii/training_steps.csv ===


FINAL: {'bleu': 11.59, 'chrf': 36.21, 'model': 'NLLB', 'setting': 'full-5epochs', 'direction': 'Kiswahili_to_Ekegusii', 'runtime_min': 40.93, 'epochs': 5, 'train_fraction': 1.0, 'freeze_encoder': True}
Full step-by-step log saved at: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/Kiswahili_to_Ekegusii/training_steps.csv
Full epoch-by-epoch log saved at: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/Kiswahili_to_Ekegusii/epoch_metrics.csv


In [19]:
print("=" * 80)
print("PSA MACHINE TRANSLATION -- NLLB PER-EPOCH RESULTS")
print("=" * 80)

for direction in NLLB_DIRECTIONS:
    run_dir = NLLB_OUTPUT / direction
    print(f"\n{'#'*80}")
    print(f"# {direction}")
    print(f"{'#'*80}")

    # ---- TRAIN: recovered from HF Trainer's own saved state (trainer_state.json) ----
    checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    train_found = False
    if checkpoints:
        state_file = checkpoints[-1] / "trainer_state.json"
        if state_file.exists():
            state = json.loads(state_file.read_text())
            log_history = state.get("log_history", [])
            step_rows = [r for r in log_history if "loss" in r and "eval_loss" not in r]
            if step_rows:
                sdf = pd.DataFrame(step_rows)
                sdf["epoch_int"] = sdf["epoch"].apply(lambda x: int(x) + 1 if x % 1 != 0 else int(x))
                train_by_epoch = sdf.groupby("epoch_int")["loss"].mean().round(4)
                print("\n--- TRAIN (average loss per epoch, recovered from trainer_state.json) ---")
                for ep, loss in train_by_epoch.items():
                    print(f"  Epoch {ep}: loss = {loss}")
                train_found = True
    if not train_found:
        print("\n--- TRAIN --- \n  No recoverable step history found.")

    # ---- VALIDATION (dev): genuinely per-epoch, one real eval each epoch ----
    epoch_log = run_dir / "epoch_metrics.csv"
    if epoch_log.exists():
        edf = pd.read_csv(epoch_log)
        print("\n--- VALIDATION / DEV (one real evaluation per epoch) ---")
        for _, row in edf.iterrows():
            print(f"  Epoch {int(row['epoch'])}: "
                  f"loss = {row['eval_loss']:.4f}, "
                  f"bleu = {row['eval_bleu']:.2f}, "
                  f"chrf = {row['eval_chrf']:.2f}")
    else:
        print("\n--- VALIDATION / DEV --- \n  No epoch log found yet.")

    # ---- TEST: only ONE result exists, by design -- never per-epoch ----
    test_log = run_dir / "final_test_metrics.csv"
    if test_log.exists():
        tdf = pd.read_csv(test_log)
        print("\n--- TEST (final, held-out -- evaluated ONCE after all training, not per epoch) ---")
        print(f"  bleu = {tdf['bleu'].iloc[0]:.2f}, chrf = {tdf['chrf'].iloc[0]:.2f}, "
              f"runtime_min = {tdf['runtime_min'].iloc[0]:.1f}")
    else:
        print("\n--- TEST --- \n  Not yet available -- training hasn't completed for this direction.")

print("\n" + "=" * 80)

PSA MACHINE TRANSLATION -- NLLB PER-EPOCH RESULTS

################################################################################
# English_to_Ekegusii
################################################################################

--- TRAIN (average loss per epoch, recovered from trainer_state.json) ---
  Epoch 1: loss = 7.3034
  Epoch 2: loss = 4.6449
  Epoch 3: loss = 4.117
  Epoch 4: loss = 3.8655
  Epoch 5: loss = 3.7514

--- VALIDATION / DEV (one real evaluation per epoch) ---
  Epoch 1: loss = 2.3526, bleu = 6.69, chrf = 29.85
  Epoch 2: loss = 2.0035, bleu = 9.18, chrf = 33.83
  Epoch 3: loss = 1.8704, bleu = 10.70, chrf = 35.50
  Epoch 4: loss = 1.8129, bleu = 11.53, chrf = 36.39
  Epoch 5: loss = 1.7939, bleu = 11.67, chrf = 36.53

--- TEST (final, held-out -- evaluated ONCE after all training, not per epoch) ---
  bleu = 11.23, chrf = 35.91, runtime_min = 40.8

################################################################################
# Kiswahili_to_Ekegusii
######

## 12. Ablation study — domain adaptation

Breaks down each fine-tuned NLLB direction's performance by PSA domain, using the same
technique and `min_examples=5` guard as the mT5 notebook, so the two are directly comparable.

In [18]:
def nllb_domain_adaptation_ablation(min_examples=5):
    out_path = RESULTS_DIR/"nllb_domain_adaptation.csv"
    if out_path.exists():
        print("  domain-adaptation ablation already computed -- loading saved result.")
        return pd.read_csv(out_path)

    rows=[]
    for direction in NLLB_DIRECTIONS:
        src,tgt=direction.split("_to_")
        best_dir=NLLB_OUTPUT/direction/"best"
        if not (best_dir/"config.json").exists():
            print(f"  {best_dir} not trained yet -- run the few-shot training cell first.")
            return None
        df=pd.read_csv(DATA_DIR/f"{direction}.test.csv").dropna(subset=["src_text","tgt_text"])
        if "Domain" not in df.columns:
            print("  no Domain column in test data -- skipping domain ablation."); return None

        tok=AutoTokenizer.from_pretrained(best_dir)
        model=AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(DEVICE)
        for dom in df["Domain"].unique():
            sub=df[df["Domain"]==dom]
            if len(sub) < min_examples:
                continue
            tok.src_lang=NLLB_CODE[src]
            preds=[]
            for i in range(0,len(sub),8):
                batch=list(sub["src_text"].iloc[i:i+8])
                enc=tok(batch,return_tensors="pt",padding=True,truncation=True,max_length=MAX_LEN).to(model.device)
                forced=tok.convert_tokens_to_ids(NLLB_CODE[tgt])
                gen=model.generate(**enc,forced_bos_token_id=forced,max_length=MAX_LEN)
                preds.extend(tok.batch_decode(gen,skip_special_tokens=True))
            bleu=sacrebleu.corpus_bleu(preds,[sub["tgt_text"].tolist()]).score
            chrf=sacrebleu.corpus_chrf(preds,[sub["tgt_text"].tolist()],word_order=2).score
            rows.append({"direction":direction,"domain":dom,"bleu":round(bleu,2),
                        "chrf":round(chrf,2),"n_examples":len(sub)})
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    domain_df=pd.DataFrame(rows)
    domain_df.to_csv(out_path,index=False)
    return domain_df

nllb_domain_df = nllb_domain_adaptation_ablation()
if nllb_domain_df is not None:
    display(nllb_domain_df)

,direction,domain,bleu,chrf,n_examples
0,English_to_Ekegusii,Health,18.44,44.00,396
1,English_to_Ekegusii,Agriculture,7.47,31.88,1401
2,English_to_Ekegusii,Education,16.36,41.92,394
3,English_to_Ekegusii,Security & Safety,12.09,35.21,388
4,English_to_Ekegusii,Governance,10.38,35.28,349
5,Kiswahili_to_Ekegusii,Health,20.07,44.46,396
6,Kiswahili_to_Ekegusii,Agriculture,8.06,32.30,1401
7,Kiswahili_to_Ekegusii,Education,14.57,40.56,394
8,Kiswahili_to_Ekegusii,Security & Safety,12.86,36.49,388
9,Kiswahili_to_Ekegusii,Governance,11.27,35.86,349


## 14. COMET on final NLLB outputs

COMET is run **after** training so it does not slow every training epoch. BLEU/chrF++ are already computed per epoch for development monitoring.

## COMET install (isolated, run only when you reach this point)

`unbabel-comet` depends on an old `pytorch-lightning` pin with malformed metadata that recent
`pip` rejects. Installing it in its own cell -- separate from the core packages -- means this
can never block training. If it fails again, tell me the exact error before retrying.

In [22]:
USE_GRADIENT_CHECKPOINTING = False   # was True

In [9]:
import subprocess, sys
try:
    import comet  # noqa: F401
    print("comet already installed.")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pip<24.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "backports.tarfile"], check=True)
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], capture_output=True, text=True)
    print("unbabel-comet installed." if result.returncode == 0 else result.stderr[-1500:])

try:
    import pytorch_lightning.utilities.argparse
except (ModuleNotFoundError, ImportError):
    pass

if "comet_model" not in dir():
    from comet import download_model, load_from_checkpoint
    comet_ckpt = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(comet_ckpt)
    print("COMET model loaded.")

comet already installed.


In [22]:
import transformers
print(transformers.__version__)

5.14.1


In [34]:
# def add_comet_to_nllb():
#     rows = []
#     for direction in NLLB_DIRECTIONS:
#         src, tgt = direction.split("_to_")
#         best = NLLB_OUTPUT / direction / "best"
#         tok = AutoTokenizer.from_pretrained(best)
#         model = AutoModelForSeq2SeqLM.from_pretrained(best).to(DEVICE)
#         ds = load_direction(direction, fewshot=None)["test"]
#         preds, refs, srcs, _ = evaluate_model(model, tok, ds, src, tgt)
#         data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(srcs, preds, refs)]
#         print(f"[{direction}] Running COMET on {len(data)} test examples (CPU)...")
#         out = comet_model.predict(data, batch_size=8, gpus=0, num_workers=0)
#         row = {"direction": direction, "model": "NLLB", "setting": "few-shot",
#               "comet": round(float(out.system_score) * 100, 2)}
#         rows.append(row)
#         print(f"[{direction}] COMET = {row['comet']}")
#         del model
#         if torch.cuda.is_available(): torch.cuda.empty_cache()
#     pd.DataFrame(rows).to_csv(RESULTS_DIR / "nllb_comet.csv", index=False)
#     display(pd.DataFrame(rows))

# add_comet_to_nllb()

## 13. Demo — quick inference on sample PSAs

Satisfies "Week 3 deliverable: working translation demo." Covers NLLB's two directions
(English->Ekegusii, Kiswahili->Ekegusii via the repurposed `kin_Latn` tag). Kept in sync
with the standalone `translate_psa.py` CLI script.

In [25]:
def demo_translate_nllb(text, src_lang, tgt_lang="Ekegusii"):
    best_dir = NLLB_OUTPUT/f"{src_lang}_to_{tgt_lang}"/"best"
    if not (best_dir/"config.json").exists():
        return f"(no trained checkpoint found for {src_lang}->{tgt_lang} yet)"
    tok=AutoTokenizer.from_pretrained(best_dir)
    model=AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(DEVICE)
    tok.src_lang=NLLB_CODE[src_lang]
    enc=tok(text,return_tensors="pt",truncation=True,max_length=MAX_LEN).to(model.device)
    forced=tok.convert_tokens_to_ids(NLLB_CODE[tgt_lang])
    gen=model.generate(**enc,forced_bos_token_id=forced,max_length=MAX_LEN)
    out=tok.decode(gen[0],skip_special_tokens=True)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out

_demo_samples = [
    ("English", "Farmers are urged to plant early this season due to expected rainfall."),
    ("Kiswahili", "Wizara ya Afya inawahimiza wakazi kukamilisha chanjo yao kabla ya Ijumaa."),
]
for src_lang, text in _demo_samples:
    print(f"[{src_lang} -> Ekegusii]")
    print(" IN :", text)
    print(" OUT:", demo_translate_nllb(text, src_lang))
    print()

[English -> Ekegusii]
 IN : Farmers are urged to plant early this season due to expected rainfall.


Predicting DataLoader 0:   0%|          | 0/366 [16:57<?, ?it/s]


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

 OUT: Abakoriria abakoriria amagoro hii kwa kumiriria kwa embura yasiyotara

[Kiswahili -> Ekegusii]
 IN : Wizara ya Afya inawahimiza wakazi kukamilisha chanjo yao kabla ya Ijumaa.


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

 OUT: Ewizara y'Oborwaria nigo egosaba abanto bonsi goika bagende barenge ase ogosusura kwabo mbere yechitariki chiomotienyi 5.



## 15. Failure recovery

If the runtime disconnects, **do not delete the checkpoint folders**.

1. Reconnect to the same Drive.
2. Set `RESUME_CHECKPOINT` to the latest folder, e.g. `.../checkpoint-750`.
3. Rerun the training cell.

Because `save_strategy="epoch"` is enabled, every completed epoch has a recoverable checkpoint.

Also inspect:
- `epoch_metrics.csv`
- `epoch_01_metrics.json`
- `epoch_02_metrics.json`
- `epoch_03_metrics.json`
- `checkpoint-*`

In [26]:
# Find the latest NLLB checkpoint for each direction:
for direction in NLLB_DIRECTIONS:
    cps=sorted((NLLB_OUTPUT/direction).glob("checkpoint-*"),key=lambda p:int(p.name.split("-")[-1]))
    print(direction, "latest:", cps[-1] if cps else "none")

English_to_Ekegusii latest: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/English_to_Ekegusii/checkpoint-3660
Kiswahili_to_Ekegusii latest: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/nllb/Kiswahili_to_Ekegusii/checkpoint-3660


In [27]:
# =========================================================================
# RESOURCE CHECK -- run this before committing to the full multi-epoch run
# =========================================================================
import subprocess

print("=== GPU MEMORY ===")
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    free = total_vram - reserved
    print(f"  Total VRAM:      {total_vram:.1f} GB")
    print(f"  Currently used:  {allocated:.1f} GB (allocated) / {reserved:.1f} GB (reserved by PyTorch)")
    print(f"  Free for us:     ~{free:.1f} GB")
else:
    print("  CUDA not available -- can't check VRAM.")

print("\n=== ACTUAL GPU-WIDE USAGE (includes other apps on this shared node) ===")
result = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                        "--format=csv"], capture_output=True, text=True)
print(" ", result.stdout.strip().replace("\n", "\n  "))

print("\n=== DISK SPACE ===")
result = subprocess.run(["df", "-h", str(Path.cwd())], capture_output=True, text=True)
print(" ", result.stdout.strip().replace("\n", "\n  "))

print("\n=== HOW MUCH SPACE OUR OUTPUTS ARE ALREADY USING ===")
for folder in [MODELS_DIR, RESULTS_DIR, LOGS_DIR, MLFLOW_DIR]:
    if folder.exists():
        size_bytes = sum(f.stat().st_size for f in folder.rglob("*") if f.is_file())
        print(f"  {folder.name}: {size_bytes / 1024**2:.1f} MB")

print("\n=== ROUGH TIME BUDGET, BASED ON YOUR EARLIER RUN ===")
print("  Your first 4-epoch, batch=8, capped run took ~53.5 min for ONE direction.")
print("  NLLB needs 2 directions total -- roughly ~1.8 hours for both at those settings.")
print("  Scaling to 6 epochs + full dataset + batch=32 changes this substantially --")
print("  use the 1-epoch quick-check's reported runtime_min x 6 x 2 directions for a")
print("  realistic estimate of the actual full run, rather than guessing from this one.")

=== GPU MEMORY ===
  Total VRAM:      79.2 GB
  Currently used:  2.3 GB (allocated) / 2.4 GB (reserved by PyTorch)
  Free for us:     ~76.9 GB

=== ACTUAL GPU-WIDE USAGE (includes other apps on this shared node) ===
  memory.used [MiB], memory.total [MiB], utilization.gpu [%]
  50429 MiB, 81920 MiB, 78 %

=== DISK SPACE ===
  Filesystem      Size  Used Avail Use% Mounted on
  overlay         968G  158G  711G  19% /

=== HOW MUCH SPACE OUR OUTPUTS ARE ALREADY USING ===
  models: 47055.0 MB
  results: 0.0 MB
  logs: 0.0 MB
  mlflow: 2.6 MB

=== ROUGH TIME BUDGET, BASED ON YOUR EARLIER RUN ===
  Your first 4-epoch, batch=8, capped run took ~53.5 min for ONE direction.
  NLLB needs 2 directions total -- roughly ~1.8 hours for both at those settings.
  Scaling to 6 epochs + full dataset + batch=32 changes this substantially --
  use the 1-epoch quick-check's reported runtime_min x 6 x 2 directions for a
  realistic estimate of the actual full run, rather than guessing from this one.


In [31]:
print("=" * 80)
print("PSA MACHINE TRANSLATION -- NLLB RESULTS (VALIDATION + TEST)")
print("=" * 80)

for direction in NLLB_DIRECTIONS:
    run_dir = NLLB_OUTPUT / direction
    print(f"\n{'#'*80}")
    print(f"# {direction}")
    print(f"{'#'*80}")

    # ---- VALIDATION (dev): genuinely per-epoch, one real eval each epoch ----
    epoch_log = run_dir / "epoch_metrics.csv"
    if epoch_log.exists():
        edf = pd.read_csv(epoch_log)
        print("\n--- VALIDATION / DEV (one real evaluation per epoch) ---")
        for _, row in edf.iterrows():
            print(f"  Epoch {int(row['epoch'])}: "
                  f"loss = {row['eval_loss']:.4f}, "
                  f"bleu = {row['eval_bleu']:.2f}, "
                  f"chrf = {row['eval_chrf']:.2f}")
    else:
        print("\n--- VALIDATION / DEV --- \n  No epoch log found.")

    # ---- TEST: only ONE result exists, by design -- never per-epoch ----
    test_log = run_dir / "final_test_metrics.csv"
    if test_log.exists():
        tdf = pd.read_csv(test_log)
        print("\n--- TEST (final, held-out -- evaluated ONCE, using the best checkpoint) ---")
        print(f"  bleu = {tdf['bleu'].iloc[0]:.2f}, chrf = {tdf['chrf'].iloc[0]:.2f}, "
              f"runtime_min = {tdf['runtime_min'].iloc[0]:.1f}")
    else:
        print("\n--- TEST --- \n  Not yet available.")

print("\n" + "=" * 80)

PSA MACHINE TRANSLATION -- NLLB RESULTS (VALIDATION + TEST)

################################################################################
# English_to_Ekegusii
################################################################################

--- VALIDATION / DEV (one real evaluation per epoch) ---
  Epoch 1: loss = 2.3526, bleu = 6.69, chrf = 29.85
  Epoch 2: loss = 2.0035, bleu = 9.18, chrf = 33.83
  Epoch 3: loss = 1.8704, bleu = 10.70, chrf = 35.50
  Epoch 4: loss = 1.8129, bleu = 11.53, chrf = 36.39
  Epoch 5: loss = 1.7939, bleu = 11.67, chrf = 36.53

--- TEST (final, held-out -- evaluated ONCE, using the best checkpoint) ---
  bleu = 11.23, chrf = 35.91, runtime_min = 40.8

################################################################################
# Kiswahili_to_Ekegusii
################################################################################

--- VALIDATION / DEV (one real evaluation per epoch) ---
  Epoch 1: loss = 2.3823, bleu = 6.96, chrf = 29.84
  Epoch 2: 

## 16. Artifact contract for Notebook 2

Notebook 2 must use the exact artifacts produced here:

```text
PSA-MT-Project/
├── shared_artifacts/
│   ├── split_manifest.json
│   └── data_processed/
│       ├── English_to_Kiswahili.train.csv
│       ├── English_to_Kiswahili.dev.csv
│       ├── English_to_Kiswahili.test.csv
│       └── ... all six directions
├── models/
│   ├── base_pretrained/
│   │   └── nllb-200-distilled-600M/
│   └── fine_tuned/
│       └── nllb/
└── results/
```

These artifacts are deliberately separated from the mT5 notebook so later deployment can load either model independently.